In [1]:
import numpy as np
import os
import re
import astropy.units as u
from astropy.wcs import WCS
from astropy.io import fits
from astropy.time import Time
from astropy.table import Table, vstack, hstack
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord, match_coordinates_sky, GeocentricTrueEcliptic
from astropy.visualization import ZScaleInterval
from skyfield.api import load, wgs84
from skyfield.api import N as North_deg
from skyfield.api import E as East_deg
from skyfield.data import mpc
from skyfield.api import Topos
from skyfield.constants import GM_SUN_Pitjeva_2005_km3_s2 as GM_SUN
from astroquery.jplhorizons import Horizons
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import warnings
warnings.filterwarnings("ignore")

In [2]:
def data_loading(MPCORB_place):

    with load.open(MPCORB_place) as f:
        minor_planets = mpc.load_mpcorb_dataframe(f)

    bad_orbits = minor_planets.semimajor_axis_au.isnull()
    minor_planets = minor_planets[~bad_orbits][2:]  # 去除无效轨道

    asteroid_names = minor_planets['designation'].to_numpy()
    packed_date = minor_planets['epoch_packed'].to_numpy()

    # 转换所有列类型为 float
    minor_planets['semimajor_axis_au'] = minor_planets['semimajor_axis_au'].astype(float)
    minor_planets['eccentricity'] = minor_planets['eccentricity'].astype(float)
    minor_planets['mean_anomaly_degrees'] = minor_planets['mean_anomaly_degrees'].astype(float)
    minor_planets['inclination_degrees'] = minor_planets['inclination_degrees'].astype(float)
    minor_planets['longitude_of_ascending_node_degrees'] = minor_planets['longitude_of_ascending_node_degrees'].astype(float)
    minor_planets['argument_of_perihelion_degrees'] = minor_planets['argument_of_perihelion_degrees'].astype(float)
    minor_planets['mean_daily_motion_degrees'] = minor_planets['mean_daily_motion_degrees'].astype(float)
    minor_planets['magnitude_H'] = minor_planets['magnitude_H'].astype(float)
    minor_planets['magnitude_G'] = minor_planets['magnitude_G'].astype(float)

    #文件内的mjd是一个特殊的压缩格式日期，我们将之转化为常用的mjd
    unzip_pack = packed_date.astype('U5')  # 转换为 Unicode 类型 U5（每个元素占5字符）
    new_unzip = unzip_pack.view('U1').reshape(-1, 5)  # 拆分为单个字符组成的二维数组
    num_unzip = new_unzip.astype('S1').view(np.uint8)  # 转为 ASCII 码便于计算

    after_date = np.zeros(len(packed_date))  # 初始化一个空数组用于存储转换后的数值

    # 处理年份：第一个字符表示世纪偏移
    after_date += (num_unzip[:, 0] - 55) * 1000000.  # 如 'J' -> 74 - 55 = 19 -> 19xx 年
    after_date += (num_unzip[:, 1] - 48) * 100000.   # 第二个字符表示年份十位/个位
    after_date += (num_unzip[:, 2] - 48) * 10000.    # 第三个字符表示月份

    # 处理第四个字符（月）：判断是数字还是字母
    bools_month_num = num_unzip[:, 3] < 60
    after_date[bools_month_num] += (num_unzip[:, 3][bools_month_num] - 48) * 100.
    after_date[~bools_month_num] += (num_unzip[:, 3][~bools_month_num] - 55) * 100.

    # 处理第五个字符（日）：同上处理方式
    bools_day_num = num_unzip[:, 4] < 60
    after_date[bools_day_num] += (num_unzip[:, 4][bools_day_num] - 48)
    after_date[~bools_day_num] += (num_unzip[:, 4][~bools_day_num] - 55)

    # 转换为字符串并格式化为 ISO 格式
    after_date = after_date.astype(np.int32).astype(np.str_)
    iso_dates = [f"{s[:4]}-{s[4:6]}-{s[6:8]}" for s in after_date]
    mjd = Time(iso_dates, format='iso').mjd

    return (
        mjd,
        minor_planets['mean_anomaly_degrees'].to_numpy(),
        minor_planets['eccentricity'].to_numpy(),
        minor_planets['semimajor_axis_au'].to_numpy(),
        minor_planets['longitude_of_ascending_node_degrees'].to_numpy(),
        minor_planets['inclination_degrees'].to_numpy(),
        minor_planets['argument_of_perihelion_degrees'].to_numpy(),
        minor_planets['mean_daily_motion_degrees'].to_numpy(),
        minor_planets['magnitude_H'].to_numpy(),
        minor_planets['magnitude_G'].to_numpy(),
        minor_planets,
        asteroid_names
    )

In [3]:
def asteriod_data_load(site):

    # 拼接行星历表（如 de421.bsp）和小行星轨道数据文件（如 MPCORB.DAT）的路径
    planet_dataset_place = 'de442.bsp'
    MPCORB_place = 'MPCORB_20250313.DAT'

    # 调用自定义函数 data_loading 加载小行星轨道数据（如从 MPCORB.DAT 中解析）
    loaded_data = data_loading(MPCORB_place)

    # 解析输入的观测站点信息
    # 假设输入格式为："经度,纬度,海拔"，例如："117.3,39.9,900"
    site_parts = site.split(',')
    lon = float(site_parts[0])  # 经度（东经），转换为浮点数
    lat = float(site_parts[1])  # 纬度（北纬），转换为浮点数
    ele = int(site_parts[2])    # 海拔高度，单位：米

    # 使用 Skyfield 加载行星历表（如 de421.bsp）
    planets = load(planet_dataset_place)
    earth = planets['earth']  # 获取地球的位置模型

    # 定义方向常量（假设 North_deg = 1, East_deg = 1，代表正北、正东方向）
    # 构建观测站的地理位置对象（基于WGS84椭球模型）
    Xinglong_set = earth + wgs84.latlon(lat, lon, ele)

    # 返回行星历表路径、加载的小行星数据、以及观测点坐标对象
    return planet_dataset_place, loaded_data, Xinglong_set

In [4]:
site = '117.575, 40.393, 960'
planet_dataset_place, loaded_data, Xinglong_set = asteriod_data_load(site)

In [5]:
def solve_kepler(Mss, e):

    M = np.radians(Mss % 360)  # 转换为弧度，并限制在 0~2π
    E = M.copy()  # 初始猜测设为平近点角

    for _ in range(100):  # 最多迭代100次
        delta = (M - E + e * np.sin(E)) / (1 - e * np.cos(E))  # 牛顿迭代公式
        E += delta
        if np.max(np.abs(delta)) < 1e-8:  # 收敛条件
            break

    return E

def xyz_to_radec_distance(x, y, z):

    distance = np.sqrt(x ** 2 + y ** 2 + z ** 2)  # 计算距离
    ra_radians = np.arctan2(y, x)  # 赤经（弧度）
    ra_hours = ra_radians * 12 / np.pi  # 转换为小时制（0~24）
    ra_hours = ra_hours % 24  # 保持在 0~24 范围内
    dec_radians = np.arcsin(z / distance)  # 赤纬（弧度）
    dec_degrees = np.degrees(dec_radians)  # 转换为度

    return ra_hours * 15, dec_degrees, distance  # 返回赤经（度）、赤纬（度）、距离

In [6]:
def calculate_asteroids_multi(mjd_epochs, M0ss, ess, ass, Omegass, iss, omegass, mdmdss, hss, gss, target_t, set_place):

    target_mjd = target_t.tt - 2400000.5  # 将观测时间转为 MJD

    x_earth, y_earth, z_earth = set_place.at(target_t).xyz.au  # 获取地球位置

    delta_t = target_mjd - mjd_epochs  # 计算当前时间与历元的时间差

    # 计算当前时刻的平近点角
    Mss = (M0ss + mdmdss * delta_t) % 360
    Ess = solve_kepler(Mss, ess)  # 解开普勒方程得到偏近点角 E
    niuss = 2 * np.arctan(np.sqrt((1 + ess) / (1 - ess)) * np.tan(Ess / 2))  # 真近点角

    used_rss = ass * (1 - ess ** 2) / (1 + ess * np.cos(niuss))  # 距离太阳的距离
    x_orbss = used_rss * np.cos(niuss)  # 轨道平面坐标
    y_orbss = used_rss * np.sin(niuss)
    z_orbss = np.zeros(len(x_orbss))

    # 将角度转为弧度
    Omegass = np.radians(Omegass)
    iss = np.radians(iss)
    omegass = np.radians(omegass)

    # 构建旋转矩阵 Q
    Q = np.array([
        [np.cos(Omegass) * np.cos(omegass) - np.sin(Omegass) * np.sin(omegass) * np.cos(iss),
         -np.cos(Omegass) * np.sin(omegass) - np.sin(Omegass) * np.cos(omegass) * np.cos(iss),
         np.sin(Omegass) * np.sin(iss)],
        [np.sin(Omegass) * np.cos(omegass) + np.cos(Omegass) * np.sin(omegass) * np.cos(iss),
         -np.sin(Omegass) * np.sin(omegass) + np.cos(Omegass) * np.cos(omegass) * np.cos(iss),
         -np.cos(Omegass) * np.sin(iss)],
        [np.sin(omegass) * np.sin(iss),
         np.cos(omegass) * np.sin(iss),
         np.cos(iss)]
    ])

    huangchi_theta = np.radians(23.5)  # 黄赤交角
    Huangchi = np.array([[1, 0, 0],
                         [0, np.cos(huangchi_theta), -np.sin(huangchi_theta)],
                         [0, np.sin(huangchi_theta), np.cos(huangchi_theta)]])

    Q = Q.transpose(2, 0, 1)  # 调整维度顺序
    pos = np.array([x_orbss, y_orbss, z_orbss])[:, np.newaxis, :].transpose(2, 0, 1)  # 重构坐标
    after_xyz = np.matmul(Q, pos)  # 应用轨道旋转
    after_xyz = np.matmul(Huangchi, after_xyz)  # 应用黄赤变换

    x, y, z = after_xyz[:, 0, 0], after_xyz[:, 1, 0], after_xyz[:, 2, 0]  # 提取最终坐标

    dx, dy, dz = x - x_earth, y - y_earth, z - z_earth  # 计算地心相对坐标
    result_radec_dis = xyz_to_radec_distance(dx, dy, dz)  # 转为赤道坐标系

    r = np.sqrt(x ** 2 + y ** 2 + z ** 2)  # 小行星到太阳的距离
    cos_alpha = (r ** 2 + result_radec_dis[2] ** 2 - 1) / (2 * r * result_radec_dis[2])
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))  # 相位角

    tan_half_alpha = np.tan(alpha / 2)
    phi1 = np.exp(-3.33 * (tan_half_alpha ** 0.63))
    phi2 = np.exp(-1.87 * (tan_half_alpha ** 1.22))
    phase = (1 - gss) * phi1 + gss * phi2  # 相位函数

    apparent_mag = hss + 5 * np.log10(result_radec_dis[2] * r) - 2.5 * np.log10(phase)  # 视星等

    return result_radec_dis[0], result_radec_dis[1], result_radec_dis[2], apparent_mag

In [7]:
def clean_asteroid_names(names):
    return [re.sub(r'^\(\d+\)\s*', '', name) for name in names]

In [ ]:
def total_location(mjd_time, mjd_time_1, ra1, ra2, dec1, dec2, lim_mag, planet_dataset_place, loaded_data, obs_set):

    # 加载时间尺度
    ts = load.timescale()

    # 将 MJD 时间转换为 Skyfield 的 UTC 时间对象
    iso_trans = Time(mjd_time, format='mjd').iso.split()
    ymd = iso_trans[0].split('-')
    hms = iso_trans[1].split(':')
    target_t = ts.utc(int(ymd[0]), int(ymd[1]), int(ymd[2]), int(hms[0]), int(hms[1]), float(hms[2]))

    # 计算每个小行星的位置和视星等
    ra, dec, dist, app_mag = calculate_asteroids_multi(
        loaded_data[0], loaded_data[1], loaded_data[2], loaded_data[3],
        loaded_data[4], loaded_data[5], loaded_data[6], loaded_data[7],
        loaded_data[8], loaded_data[9], target_t, obs_set
    )  # 结果包含(ra_array, dec_array, distance_array, mag_array)
    # logger.info(f"The loaded data is a {type(return_rough_location)}, with {len(return_rough_location)} elements.")

    # 筛选出符合赤经、赤纬和视星等限制的小行星
    cuts = ~((ra < ra1-1) | (ra > ra2+1) |
             (dec < dec1-1) | (dec > dec2+1) |
             (app_mag > lim_mag))
    
    # 获取符合条件的小行星信息
    need_asteroid = loaded_data[-2][cuts]
    need_names = loaded_data[-1][cuts]
    need_mags = app_mag[cuts]
    # logger.info(f'The roughly calculated asteriods with in the location has names of {need_names}, and mag {need_mags}. ')
    
    # 初始化结果存储变量
    ras = []
    decs = []
    
    # 加载星历数据
    planets = load(planet_dataset_place)
    sun = planets['sun']
    
    # 对每个符合条件的小行星进行精确计算
    for i in range(len(need_asteroid)):
        one_asteroids = sun + mpc.mpcorb_orbit(need_asteroid.iloc[i], ts, GM_SUN)
        radecdis = obs_set.at(target_t).observe(one_asteroids).radec()
        ras.append(radecdis[0].degrees)
        decs.append(radecdis[1].degrees)
        
    """
    cleaned_names = clean_asteroid_names(need_names)

    # 返回结果
    for i in range(len(need_mags)):
        designation = cleaned_names[i]
        obs_time = mjd_time_1
        obj = Horizons(id=designation, id_type='smallbody', location='327', epochs=obs_time.jd)
        eph = obj.ephemerides()
        coord = SkyCoord(ra=eph['RA'][0]*u.deg, dec=eph['DEC'][0]*u.deg, frame='icrs')
        ras.append(coord.ra.deg)
        decs.append(coord.dec.deg)
    """
    
    # 创建结构化数组
    dtype = [('name', object), ('ra', float), ('dec', float), ('mag', float)]
    result_array = np.zeros(len(need_names), dtype=dtype)
    result_array['name'] = need_names
    result_array['ra'] = ras
    result_array['dec'] = decs
    result_array['mag'] = need_mags
    return result_array

In [13]:
cat_dir = '/Volumes/Foundation/SMT_data/test'
lim_mag = 21
sep_limit = 3 * u.arcsec
obs_set = Xinglong_set

all_asteroids = []
all_matches = []

for fname in os.listdir(cat_dir):
    if not fname.endswith('.fits'):
        continue

    fpath = os.path.join(cat_dir, fname)
    try:
        hdul = fits.open(fpath)
        header = hdul[0].header
        cat = Table(hdul[1].data)
        w = WCS(header)

        ny, nx = 9576, 6388
        # 图像四个角的像素坐标
        pix_corners = [[1, 1], [1, ny], [nx, 1], [nx, ny]]
        world = w.all_pix2world(pix_corners, 1)  # RA/DEC in degrees

        ra_vals = world[:, 0]
        dec_vals = world[:, 1]
        ra1, ra2 = ra_vals.min(), ra_vals.max()
        dec1, dec2 = dec_vals.min(), dec_vals.max()

        mjd_time = header['JD'] - 2400000.5  # MJD
        mjd_time_1 = Time(header['DATE-OBS'])

    except Exception as e:
        print(f"Failed to load {fname}: {e}")
        continue

    # 获取可能的小行星
    try:
        result_array = total_location(
            mjd_time, mjd_time_1, ra1, ra2, dec1, dec2, lim_mag,
            planet_dataset_place=planet_dataset_place, loaded_data=loaded_data, obs_set=obs_set
        )
    except Exception as e:
        print(f"total_location failed on {fname}: {e}")
        continue
    
    names, ras, decs, mags = result_array['name'].astype('<U24'), result_array['ra'], result_array['dec'], result_array['mag']
    
    # 将精算出的小行星赤道坐标转换为像素坐标
    pix_coords = w.all_world2pix(ras, decs, 1)

    x_pix, y_pix = pix_coords  # 转置为 x, y 列

    # 筛选出落在图像范围内的点（WCS 是 1-based）
    inside_mask = (x_pix >= 1) & (x_pix <= nx) & (y_pix >= 1) & (y_pix <= ny)

    # 如果都在视场外就跳过
    if not np.any(inside_mask):
        print(f"All asteroids in {fname} are out of image bounds.")
        continue

    # 根据 mask 筛选保留下来的小行星
    names = names[inside_mask]
    ras = ras[inside_mask]
    decs = decs[inside_mask]
    mags = mags[inside_mask]
    fname_list = [fname] * len(names)

    asteroid_tbl = Table([names, ras, decs, mags, fname_list], names=('name', 'ra', 'dec', 'mag', 'source_file'))
    all_asteroids.append(asteroid_tbl)
    
    """
     # === Cutout 操作 ===
    try:
        # 图像路径和输出目录准备
        image_path = os.path.join('./near_img', fname.replace('_cat', ''))
        if not os.path.exists(image_path):
            print(f"Image file not found for cutout: {image_path}")
        else:
            with fits.open(image_path) as hdul_img:
                img_data = hdul_img[0].data
                w_img = WCS(hdul_img[0].header)

                # 输出文件夹
                base_name = os.path.splitext(fname)[0].replace('_cat', '')
                out_dir = os.path.join('./cutouts', base_name)
                os.makedirs(out_dir, exist_ok=True)

                # RA/DEC → pixel 坐标
                asteroid_coords = SkyCoord(ras * u.deg, decs * u.deg)
                x_pixs, y_pixs = w_img.world_to_pixel(asteroid_coords)

                # Zscale
                interval = ZScaleInterval()

                for i, (x_pix, y_pix, row) in enumerate(zip(x_pixs, y_pixs, asteroid_tbl)):
                    try:
                        cutout = Cutout2D(img_data, (x_pix, y_pix), (300, 300), wcs=w_img)
                        cut_data = cutout.data
                        vmin, vmax = interval.get_limits(cut_data)
                        norm_data = (cut_data - vmin) / (vmax - vmin)
                        norm_data = norm_data.clip(0, 1)

                        # 输出路径
                        cutout_name = f'{i:03d}_{row["name"].replace(" ", "_")}.png'
                        cutout_path = os.path.join(out_dir, cutout_name)

                        # 画图并添加绿色圆圈
                        fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
                        ax.imshow(norm_data, cmap='gray', origin='lower')
                        
                        # 圆心位置：cutout 中心总是 (150, 150)
                        circ = Circle((150, 150), radius=10, edgecolor='lime', facecolor='none', linewidth=1.5)
                        ax.add_patch(circ)

                        ax.axis('off')
                        plt.tight_layout(pad=0)
                        plt.savefig(cutout_path, bbox_inches='tight', pad_inches=0)
                        plt.close(fig)
                    except Exception as e:
                        print(f"Cutout failed for {row['name']}: {e}")
    except Exception as e:
        print(f"General cutout error on {fname}: {e}")
    """
    # 进行交叉匹配
    try:
        asteroid_coords = SkyCoord(ras * u.deg, decs * u.deg)
        cat_coords = SkyCoord(cat['RA'] * u.deg, cat['DEC'] * u.deg)

        idx, sep2d, _ = match_coordinates_sky(asteroid_coords, cat_coords)
        matched = sep2d < sep_limit

        if any(matched):
            matched_ast = asteroid_tbl[matched]
            matched_cat = cat[idx[matched]]

            combined = hstack([matched_ast, matched_cat], join_type='exact')
            all_matches.append(combined)
    except Exception as e:
        print(f"Matching error on {fname}: {e}")
        continue

    print(f"Processed {fname}: {len(asteroid_tbl)} candidates, {sum(matched)} matched")

# 写入最终合并结果
if all_asteroids:
    total_ast = vstack(all_asteroids)
    total_ast.write(os.path.join(cat_dir, 'asteroids_all.fits'), overwrite=True)
    print(f"Saved total asteroid catalog: {len(total_ast)} entries")

if all_matches:
    total_match = vstack(all_matches)
    total_match.write(os.path.join(cat_dir, 'matched_sources.fits'), overwrite=True)
    print(f"Saved matched sources catalog: {len(total_match)} matches")
else:
    print("No matched sources found.")

Processed OBJ_0029P_0231_img_cat.fits: 20 candidates, 1 matched
Saved total asteroid catalog: 20 entries
Saved matched sources catalog: 1 matches
